In [ ]:
# Some basics
import cv2
import math
import os
import pandas as pd
import numpy as np
import time

# Declare filenames
ROOT_DATA_DIR = (r"~\Data\Eyetracking_03_Matched_frames") # folder where the Input data sits
DATA_DIR_OUTPUT =  (r"~\Data\Eyetracking_04_Match_Segmentation")
SEMANTIC_SEGMENTATION_DIR = (r'~\Data\Semantic_01_Segmentation\Robust_batch')

# Define the column names for each type of eyetracking data
fixation_columns = ['tStart',	'tEnd',	'duration',	'xAvg',	'yAvg',	'pupilAvg',	'Movie', 'matched_frame_IDx_start',	'matched_frame_IDx_end']
saccade_columns  = ['tStart', 'tEnd', 'duration', 'xStart', 'yStart', 'xEnd', 'yEnd', 'ampDeg', 'vPeak', 'Movie', 'matched_frame_IDx_start', 'matched_frame_IDx_end']
sample_columns   = ['tSample', 'LX', 'LY', 'LPupil', 'RX', 'RY', 'RPupil', 'Movie', 'matched_frame_IDx']

# Define a dictionary to map movie labels to their corresponding video files
movie_files = {
    "movie_01": "Charite",
    "movie_02": "Ziemlich_Beste_Freunde",
    "movie_03": "High_Seas",
    "movie_04": "Biohackers",
    "movie_05": "Downton_Abbey",
    "movie_06": "New_Amsterdam"
}

# Define a mapping of eyetracking types to coordinates
coordinates_mapping = {
    "Fixation": ['xAvg', 'yAvg'],
    "Saccade": ['xStart', 'yStart', 'xEnd', 'yEnd'],
    "Samples": ['LX', 'LY', 'RX', 'RY']
}

frame_width = 1921
frame_height = 1081


# Iterate through each eye tracking file
for participant_folder in os.listdir(ROOT_DATA_DIR):
    
    # Construct the full path to the participant subfolder
    participant_folder_path = os.path.join(ROOT_DATA_DIR, participant_folder)
    print('Participant Directory:', participant_folder_path)

    # Check if the subfolder is a directory
    if os.path.isdir(participant_folder_path):
        
        for eyetracking_file in os.listdir(participant_folder_path):  # Assuming your eye tracking files have a .csv extension
            
            # Construct the output file path
            output_file = os.path.join(DATA_DIR_OUTPUT, f'{eyetracking_file[:-19]}_Pixel_Category.csv')
            print(f'OutputFile', output_file)
            output_file_path = os.path.join(DATA_DIR_OUTPUT, output_file)

            # Check if the output file already exists
            if os.path.exists(output_file_path):
                #print(f"Output file already exists for {eyetracking_file}. Skipping to the next file.")
                continue
            
            file_path = os.path.join(participant_folder_path, eyetracking_file)
            movie_name = None

            # Extract the information from the eyetracking file name
            file_parts = eyetracking_file[:-3].split("_")
            participant = file_parts[0] 
            block_number = file_parts[-7]
            order_number = file_parts[-5]
            eyetracking_type = file_parts[-9]

            # Determine the column structure based on the eyetracking type
            if eyetracking_type == "Fixation":
                eyetracking_columns = fixation_columns
            elif eyetracking_type == "Saccade":
                eyetracking_columns = saccade_columns
            elif eyetracking_type == "Samples":
                eyetracking_columns = sample_columns
            else:
                print(f"Unknown eyetracking type in file: {eyetracking_file}")
                continue

            # Load the eye tracking data
            eyetracking_data = pd.read_csv((file_path), delimiter=',')
            print(f'File Loaded: {file_path} ...')
            
            # Extract the desired columns from the eyetracking data if they exist in the file
            eyetracking_columns_present = [col for col in eyetracking_columns if col in eyetracking_data.columns]
            eyetracking_data = eyetracking_data[eyetracking_columns]
            movie_name = eyetracking_data['Movie'].loc[0]
            #print(movie_name)
        

            for mask_folder in os.listdir(SEMANTIC_SEGMENTATION_DIR):
                    if mask_folder.startswith(movie_files[movie_name][0]):
                        # Construct the full path to the participant subfolder
                        mask_folder_path = os.path.join(SEMANTIC_SEGMENTATION_DIR, mask_folder)
                        frame_length = os.listdir(mask_folder_path)
                        #print("Path to corresponding Folder:", mask_folder_path)
                        
                        # Iterate through each eye tracking data point and match with frames
                        for index, row in eyetracking_data.iterrows():

                            if eyetracking_type == "Fixation":
                                #print("Fixations found...")
                                frame_index_start = row['matched_frame_IDx_start']
                                frame_index_end = row['matched_frame_IDx_end']
                                x_avg = row['xAvg']
                                y_avg = row['yAvg']

                                # Check if x_avg is out of bounds
                                # For each eye-tracking type, it checks if the coordinates are out of bounds, 
                                # and if so, clips them to valid ranges and marks the corresponding entry with 'out of bounds'.
                                if x_avg < 0 or x_avg >= frame_width:
                                    # print('x_avg',x_avg)
                                    x_avg = np.clip(x_avg, 0, frame_width - 1)  # Clip to valid range
                                    eyetracking_data.at[index, 'MatchingPixel_Category'] = 'out of bounds'

                                # Check if y_avg is out of bounds
                                if y_avg < 0 or y_avg >= frame_height:
                                    y_avg = np.clip(y_avg, 0, frame_height - 1)  # Clip to valid range
                                    eyetracking_data.at[index, 'MatchingPixel_Category'] = 'out of bounds'
                                
                                # Extract the corresponding frames based on the frame indices
                                if frame_index_start < 0 or frame_index_start >= (len(frame_length)/2):
                                    print(f"Frame index out of bounds: {frame_index_start}. Skipping this data point.")
                                    continue

                                if frame_index_end < 0 or frame_index_end >= (len(frame_length)/2):
                                    print(f"Frame index end out of bounds: {frame_index_end}. Skipping this data point.")
                                    continue

                                # Load the corresponding frame_data for start frame index
                                frame_filename_start = f"{mask_folder}_frame_{frame_index_start:04}.csv"
                                frame_file_path_start = os.path.join(mask_folder_path, frame_filename_start)
                                frame_data_start = pd.read_csv(frame_file_path_start, delimiter=';', header=None)
                                
                                # Load the corresponding frame_data for end frame index
                                frame_filename_end = f"{mask_folder}_frame_{frame_index_end:04}.csv"
                                frame_file_path_end = os.path.join(mask_folder_path, frame_filename_end)
                                frame_data_end = pd.read_csv(frame_file_path_end, delimiter=';', header=None)

                                
                                # Use x_avg and y_avg to get pixel indices for start and end frames np.floor() rounds it down to the nearest integer
                                x_index_start = math.floor(x_avg)
                                y_index_start = math.floor(y_avg)
                                x_index_end = math.floor(x_avg)
                                y_index_end = math.floor(y_avg)

                                # Get the matching pixels from frame_data_start and frame_data_end
                                matching_pixel_start = frame_data_start.iloc[y_index_start, x_index_start].astype(int)
                                matching_pixel_end = frame_data_end.iloc[y_index_end, x_index_end].astype(int)
                                
                                # Update the corresponding columns in the eyetracking data
                                eyetracking_data.at[index, 'MatchingPixel_Start'] = matching_pixel_start
                                eyetracking_data.at[index, 'MatchingPixel_End'] = matching_pixel_end

                                

                            elif eyetracking_type == "Saccade":
                                #print("Saccades found...")
                                frame_index_start = row['matched_frame_IDx_start']
                                frame_index_end = row['matched_frame_IDx_end']
                                xStart = row['xStart']
                                yStart = row['yStart']
                                xEnd = row['xEnd']
                                yEnd = row['yEnd']

                                # Check if xStart is out of bounds
                                if xStart < 0 or xStart >= frame_width:
                                    xStart = np.clip(xStart, 0, frame_width - 1)  # Clip to valid range
                                    eyetracking_data.at[index, 'MatchingPixel_Category'] = 'out of bounds'
                                # Check if yStart is out of bounds
                                if yStart < 0 or yStart >= frame_height:
                                    yStart = np.clip(yStart, 0, frame_height - 1)  # Clip to valid range
                                    eyetracking_data.at[index, 'MatchingPixel_Category'] = 'out of bounds'
                                # Check if xEnd is out of bounds
                                if xEnd < 0 or xEnd >= frame_width:
                                    xEnd = np.clip(xEnd, 0, frame_width - 1)  # Clip to valid range
                                    eyetracking_data.at[index, 'MatchingPixel_Category'] = 'out of bounds'
                                # Check if yEnd is out of bounds
                                if yEnd < 0 or yEnd >= frame_height:
                                    yEnd = np.clip(yEnd, 0, frame_height - 1)  # Clip to valid range
                                    eyetracking_data.at[index, 'MatchingPixel_Category'] = 'out of bounds'

                                # Extract the corresponding frames based on the frame indices
                                if frame_index_start < 0 or frame_index_start >= (len(frame_length)/2):
                                    print(f"Frame index start out of bounds: {frame_index_start}. Skipping this data point.")
                                    continue

                                if frame_index_end < 0 or frame_index_end >= (len(frame_length)/2):
                                    print(f"Frame index end out of bounds: {frame_index_end}. Skipping this data point.")
                                    continue

                                # Load the corresponding frame_data for start frame index
                                frame_filename_start = f"{mask_folder}_frame_{frame_index_start:04}.csv"
                                frame_file_path_start = os.path.join(mask_folder_path, frame_filename_start)
                                frame_data_start = pd.read_csv(frame_file_path_start, delimiter=';', header=None)
                                
                                # Load the corresponding frame_data for end frame index
                                frame_filename_end = f"{mask_folder}_frame_{frame_index_end:04}.csv"
                                frame_file_path_end = os.path.join(mask_folder_path, frame_filename_end)
                                frame_data_end = pd.read_csv(frame_file_path_end, delimiter=';', header=None)
                                
                                # Use x_start, y_start, x_end, y_end to get pixel indices for start and end frames
                                x_index_start = math.floor(xStart)
                                y_index_start = math.floor(yStart)
                                x_index_end = math.floor(xEnd)
                                y_index_end = math.floor(yEnd)

                                # Search for the pixels with the corresponding coordinates in frames_data for both start and end frames
                                matching_pixel_start = frame_data_start.iloc[y_index_start, x_index_start].astype(int)
                                matching_pixel_end = frame_data_end.iloc[y_index_end, x_index_end].astype(int)

                                eyetracking_data.at[index, 'MatchingPixel_Category_Start'] = matching_pixel_start
                                eyetracking_data.at[index, 'MatchingPixel_Category_End'] = matching_pixel_end

                                
                            
                            elif eyetracking_type == "Samples":
                                #print("Samples found...")
                                frame_index_start = row['matched_frame_IDx']
                                LX = row['LX']
                                LY = row['LY']
                                RX = row['RX']
                                RY = row['RY']

                                frame_filename = None  # Define an initial frame_filename
                                
                                # Check if the frame_filename has changed
                                if frame_filename != f"{mask_folder}_frame_{frame_index_start:04}.csv":
                                    frame_filename = f"{mask_folder}_frame_{frame_index_start:04}.csv"
                                    print("Frame {frame_filename} found ")
                                    frame_file_path = os.path.join(mask_folder_path, frame_filename)
                                    frame_data = pd.read_csv(frame_file_path, delimiter=';', header=None)

                                # Extract the corresponding frames based on the frame indices
                                if frame_index_start < 0 or frame_index_start >= (len(frame_length)/2):
                                    print(f"Frame index start out of bounds: {frame_index_start}. Skipping this data point.")
                                    continue
                                
                                if not pd.isnull(LX) and not pd.isnull(LY):
                                    # Use LX, LY to get pixel indices for start frame
                                    LX_index = math.floor(LX)
                                    LY_index = math.floor(LY)

                                    #Check if xStart is out of bounds
                                    if LX_index < 0 or LX_index >= frame_width:
                                        #LX_index = np.clip(LX_index, 0, frame_width - 1)  # Clip to valid range
                                        eyetracking_data.at[index, 'Issue'] = 'out of bounds'
                                    
                                    # Check if yStart is out of bounds
                                    if LY_index < 0 or LY_index >= frame_height:
                                        #LY_index = np.clip(LY_index, 0, frame_height - 1)  # Clip to valid range
                                        eyetracking_data.at[index, 'Issue'] = 'out of bounds'
                                    
                                    # Get the matching pixel from frame_data
                                    print('Pixel Category found')
                                    matching_pixel = frame_data.iloc[LY_index, LX_index].astype(int)
                                    
                                    # Update the corresponding column in the eyetracking data
                                    print('Column updated')
                                    eyetracking_data.at[index, 'MatchingPixel'] = matching_pixel

                                    
                                elif not pd.isnull(RX) and not pd.isnull(RY):
                                    # Use RX, RY to get pixel indices for start frame
                                    RX_index = math.floor(RX)
                                    RY_index = math.floor(RY)

                                    if RX_index < 0 or RX_index >= frame_width:
                                        RX_index = np.clip(RX_index, 0, frame_width - 1)  # Clip to valid range
                                        eyetracking_data.at[index, 'Issue'] = 'out of bounds'
                                    # Check if yStart is out of bounds
                                    if RY_index < 0 or RY_index >= frame_height:
                                        RY_index = np.clip(RY_index, 0, frame_height - 1)  # Clip to valid range
                                        eyetracking_data.at[index, 'Issue'] = 'out of bounds'
                                    
                                    # Get the matching pixel from frame_data
                                    matching_pixel = frame_data.iloc[RY_index, RX_index].astype(int)
                                    
                                    # Update the corresponding column in the eyetracking data
                                    eyetracking_data.at[index, 'MatchingPixel'] = matching_pixel


                            else:
                                print(f"Unknown eyetracking type in file: {eyetracking_file}")
                                continue

            eyetracking_data.to_csv(output_file_path)
            print('File Saved')
            print("Participant ", participant, "Block", block_number, "Order", order_number)
            print("------------------------------")




In [ ]:
# # Determine the movie associated with the eye tracking file
#         for movie, folder_name in movie_files.items():
#             if folder_name in file_name:
#                 print(folder_name)
#                 movie_name = movie
#                 print(movie_name)
#                 break

#         if movie_name is None:
#             continue  # Skip the file if no matching movie is found

In [ ]:
# # Iterate through each eye tracking data point and match with frames
#                         for index, row in eyetracking_data.iterrows():
#                             try:
#                                 coordinates = []
#                                 for column in coordinates_mapping[eyetracking_type]:
#                                     # Get the coordinate value from the eyetracking data
#                                     coordinate = row[column]
#                                     # Round the decimal coordinate to the nearest whole number
#                                     rounded_coordinate = round(coordinate)
#                                     coordinates.append(rounded_coordinate)

#                                 # Search for the pixel with the closest coordinates
#                                 matching_pixel = frames_data.loc[(frames_data.iloc[:, :2] == coordinates).all(axis=1)]

#                                 # Extract the relevant information from the matching frames
#                                 extracted_info = matching_pixel.iloc[:, 2:]  # Adjust the column indices as per your frames data structure

#                                 # Save the extracted information in the eye tracking data file
#                                 for i, column in enumerate(extracted_info.columns):
#                                     eyetracking_data[f'Attribute{i+1}'] = extracted_info[column]

#                             except (KeyError, IndexError) as e:
#                                 print(f"Error matching frames for coordinates: {coordinates}")
#                                 print(f"Error message: {str(e)}")

In [ ]:
# # Iterate through each coordinate mapping
#                         for coordinate_column, coordinates in coordinates_mapping.items():
#                             # Calculate the indices of the pixel corresponding to the rounded coordinates
#                             rounded_coordinates = eyetracking_data[coordinates].round().astype(int)

#                             # Check if the indices are within the frame dimensions
#                             valid_indices = (rounded_coordinates >= 0) & (rounded_coordinates < frame_width)

#                             # Update the eyetracking data with the extracted information
#                             eyetracking_data[f'{coordinate_column}_PixelValue'] = pd.NA

#                             # Calculate pixel indices for valid coordinates
#                             pixel_indices = rounded_coordinates[valid_indices].apply(lambda x: x[1] * frame_width + x[0])

#                             # Extract the relevant information from the matching pixel indices
#                             extracted_info = frames_data.iloc[pixel_indices, 2:]  # Adjust the column indices as per your frames data structure

#                             # Assign the extracted information to the eyetracking data
#                             eyetracking_data.loc[valid_indices, f'{coordinate_column}_PixelValue'] = extracted_info.values.flatten()



# elif eyetracking_type == "Saccades":
#     print("Saccades found...")
#     frame_index_start = row['matched_frame_IDx_start']
#     frame_index_end = row['matched_frame_IDx_end']
#     xStart = row['xStart']
#     yStart = row['yStart']
#     xEnd = row['xEnd']
#     yEnd = row['yEnd']

#     # Load the corresponding frame_data for start and end frame indices
#     frame_filename_start = f"{mask_folder}_frame_{frame_index_start:04}.csv"
#     frame_filename_end = f"{mask_folder}_frame_{frame_index_end:04}.csv"
#     frame_file_path_start = os.path.join(mask_folder_path, frame_filename_start)
#     frame_file_path_end = os.path.join(mask_folder_path, frame_filename_end)
#     frame_data_start = pd.read_csv(frame_file_path_start, delimiter=';', header=None)
#     frame_data_end = pd.read_csv(frame_file_path_end, delimiter=';', header=None)

#     try:
#         # Get pixel indices for start and end frames
#         xStart_index = math.floor(xStart)
#         yStart_index = math.floor(yStart)
#         xEnd_index = math.floor(xEnd)
#         yEnd_index = math.floor(yEnd)

#         # Check if pixel indices are out of bounds
#         if xStart_index < 0 or xStart_index >= frame_data_start.shape[1] or \
#            yStart_index < 0 or yStart_index >= frame_data_start.shape[0]:
#             raise IndexError("Start pixel index out of bounds")
        
#         if xEnd_index < 0 or xEnd_index >= frame_data_end.shape[1] or \
#            yEnd_index < 0 or yEnd_index >= frame_data_end.shape[0]:
#             raise IndexError("End pixel index out of bounds")

#         # Get pixel color values for start and end frames
#         start_pixel_value = frame_data_start.iloc[yStart_index, xStart_index]
#         end_pixel_value = frame_data_end.iloc[yEnd_index, xEnd_index]

#         # Calculate distance between start and end pixels
#         pixel_distance = math.sqrt((xEnd_index - xStart_index)**2 + (yEnd_index - yStart_index)**2)

#         # Update the 'Pixel_Category' column based on pixel distance
#         if pixel_distance < threshold_distance:
#             row['Pixel_Category'] = "static"
#         else:
#             row['Pixel_Category'] = "moving"
            
#     except IndexError as e:
#         row['Pixel_Category'] = "out of bounds"
#         print(f"Error: {e}")
        
#     except Exception as e:
#         row['Pixel_Category'] = "unknown"
#         print(f"Error: {e}")

#     # Save output file after processing every row
#     if i == len(eyetracking_data) - 1:
#         output_file = os.path.join(DATA_DIR_OUTPUT, f'{file_name[:-20]}_Pixel_Category.xlsx')
#         output_file_path = os.path.join(DATA_DIR_OUTPUT, output_file)
#         eyetracking_data.to_excel(output_file_path, index=False)
